In [31]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.chrome.options import Options
import time
import pandas as pd
import logging
from datetime import datetime
import numpy as np

### Configurações Iniciais

In [32]:
# Lista de produtos para pesquisa
PRODUTOS_ELETRONICOS = [
    "Caixa de Som JBL Flip 6",
    "Smart TV Samsung 50 polegadas Crystal UHD",
    # "Smartphone Xiaomi Redmi Note 15",
    # "Smartphone Samsung Galaxy S23 Ultra",
    # "Tablet Apple iPad Air M2",
    # "Notebook Gamer Acer Nitro 5",
    # "Monitor Gamer LG Ultragear 27",
    # "Mouse Sem Fio Logitech G305",
    # "Teclado Mecânico Razer BlackWidow",
    # "Headset Gamer HyperX Cloud II",
    # "Fone Bluetooth Sony WH-1000XM5",
    # "Caixa de som JBL Boombox 3",
    # "Apple Watch Series 9",
    # "Console PlayStation 5 Slim",
    # "Console Xbox Series X",
    # "Placa de Vídeo RTX 4060 NVIDIA",
    # "Memória RAM Kingston Fury 8GB DDR4",
    # "SSD Kingston NV2 1TB NVMe",
    # "Processador Intel Core i5-13400F",
    # "Webcam Logitech C920 Full HD",
    # "Roteador Wi-Fi 6 TP-Link Archer",
    # "Impressora Epson EcoTank L3250",
    # "Carregador Portátil Baseus 20000mAh",
    # "Cabo HDMI 2.1 8K Baseus",
    # "Microfone Condensador HyperX QuadCast",
    # "Suporte para Monitor articulado F80N"
]

# Armazenamento dos resultados
lista_produtos = []


options = Options()

# Remove a flag "navigator.webdriver = true" que identifica o Selenium
options.add_argument("--disable-blink-features=AutomationControlled")

# Remove o banner "Chrome está sendo controlado por software automatizado"
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

# Simula um usuário real
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

navegador = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

# Oculta o webdriver via JavaScript logo após abrir o navegador
navegador.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

# Configuração do Navegador
navegador.maximize_window()

nome_arquivo_log = f"log_execucao_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

logging.basicConfig(
    filename=nome_arquivo_log,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%d/%m/%Y %H:%M:%S",
    encoding="utf-8"
)

logging.info("="*60)
logging.info("INÍCIO DA EXECUÇÃO")
logging.info("="*60)


# URL de referência
url_kabum = "https://www.kabum.com.br"
url_amazon = "https://www.amazon.com.br"
url_mercado_livre = "https://www.mercadolivre.com.br"

### Web Scraping - Kabum

In [33]:
logging.info(f"Acessando site: {url_kabum}")
navegador.get(url_kabum)

WebDriverWait(navegador, 10).until(
    EC.element_to_be_clickable(
        (By.ID, "inputBusca") 
    )
)

for item_pesquisa in PRODUTOS_ELETRONICOS:
    try:
        print(f"\n>>> Pesquisando por: {item_pesquisa}...")
        logging.info(f"Pesquisando produto: {item_pesquisa}")
        
        time.sleep(1) 
        
        busca = navegador.find_element("xpath", "//*[@id='inputBusca']")
        
        busca.clear()
        
        busca.send_keys(item_pesquisa)
        time.sleep(1) 
        busca.send_keys(Keys.ENTER)

        WebDriverWait(navegador, 10).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, ".desktop\:my-8")
            )
        )
        

        nomes_elementos = navegador.find_elements("class name", "h-40")[:3]
        
        precos_elementos = navegador.find_elements(
            "xpath", 
            "//div[contains(@class, 'flex gap-4 items-center')]/span[2]"
        )[:3]

        links_elementos = navegador.find_elements(
            "css selector",
            "a.flex.flex-col.relative.gap-4"
        )[:3]

        for i in range(len(nomes_elementos)):
            try:
                nome = nomes_elementos[i].text
                
                if i < len(precos_elementos):
                    texto_preco = precos_elementos[i].text.replace("R$", "").strip()
                    valor = f"R$ {texto_preco}" if texto_preco else "Sem preço"
                else:
                    valor = "Sem preço"
                # -----------------------------------
                
                link = links_elementos[i].get_attribute("href") if i < len(links_elementos) else "Sem link"
                
                lista_produtos.append({
                    "Pesquisa": item_pesquisa,
                    "Produto": nome,
                    "Preço": valor,
                    "Link": link,
                    "Loja": "KaBuM"
                })
                
                print(f"Encontrado: {nome[:50]}... | Valor: {valor}")

                logging.info(
                    f"[KaBuM] Produto encontrado | "
                    f"Nome: {nome} | "
                    f"Preço: {valor}"
                )
                
            except Exception as e:
                 erro = f"Erro ao processar item {i}: {str(e)}"
                 print(erro)
                 logging.error(erro)

    except Exception as e:
        erro = f"Erro na busca '{item_pesquisa}': {str(e)}"
        print(erro)
        logging.error(erro)
        time.sleep(3)

print("\n>>> Pesquisa concluída. Indo para próxima etapa...")

<>:27: SyntaxWarning: "\:" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\:"? A raw string is also an option.
<>:27: SyntaxWarning: "\:" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\:"? A raw string is also an option.
C:\Users\ricar\AppData\Local\Temp\ipykernel_16868\7458116.py:27: SyntaxWarning: "\:" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\:"? A raw string is also an option.
  (By.CSS_SELECTOR, ".desktop\:my-8")



>>> Pesquisando por: Caixa de Som JBL Flip 6...
Encontrado: Caixa de Som Bluetooth Portátil Charge 6 JBL - Pre... | Valor: R$ 996,98
Encontrado: Caixa De Som Portátil JBL Flip 6, Bluetooth, 20W R... | Valor: R$ 1.171,01
Encontrado: Caixa De Som Portátil JBL Flip 6, Bluetooth, 20W R... | Valor: R$ 889,00

>>> Pesquisando por: Smart TV Samsung 50 polegadas Crystal UHD...
Encontrado: Samsung Smart TV 50" Crystal UHD 4K U8600F 2025, X... | Valor: R$ 2.219,90
Encontrado: Smart TV 50" Samsung UHD 4K Crystal UHD U8600F UN5... | Valor: R$ 2.768,00
Encontrado: Samsung Smart Tv 58” Crystal Uhd 4k U8500f 2025, X... | Valor: R$ 2.789,90

>>> Pesquisa concluída. Indo para próxima etapa...


### Web Scraping - Mercado Livre

In [34]:

logging.info(f"Acessando site: {url_mercado_livre}")
navegador.get(url_mercado_livre)

WebDriverWait(navegador, 10).until(
    EC.element_to_be_clickable(
        (By.ID, "cb1-edit") 
    )
)

for item_pesquisa in PRODUTOS_ELETRONICOS:
    try:
        print(f"\n>>> Pesquisando por: {item_pesquisa}...")
        logging.info(f"Pesquisando produto: {item_pesquisa}")
        
        time.sleep(0.5) 
        
        busca = navegador.find_element(By.XPATH, "//*[@id='cb1-edit']")
        
        busca.clear()
        
        busca.send_keys(item_pesquisa)
        time.sleep(1) 
        busca.send_keys(Keys.ENTER)

        WebDriverWait(navegador, 10).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "#cb1-edit")
            )
        )
        
        time.sleep(1) 

        print("Capturando nomes e links...")
        nomes_e_links = navegador.find_elements(By.CSS_SELECTOR, "a.poly-component__title")[:3]
        
        print("Capturando preços...")
        precos_elementos = navegador.find_elements(By.CLASS_NAME, "poly-component__price")[:3]

        for i in range(len(nomes_e_links)):
            try:
                nome = nomes_e_links[i].text
                link = nomes_e_links[i].get_attribute("href")
                
                try:
                    inteiro = precos_elementos[i].find_element(By.CLASS_NAME, "andes-money-amount__fraction").text
                    
                    try:
                        centavos = precos_elementos[i].find_element(By.CLASS_NAME, "andes-money-amount__cents").text
                        valor = f"R$ {inteiro},{centavos}"
                    except:
                        valor = f"R$ {inteiro},00"
                        
                except Exception as e:
                    valor = "Sem preço"

                lista_produtos.append({
                    "Pesquisa": item_pesquisa,
                    "Produto": nome,
                    "Preço": valor,
                    "Link": link,
                    "Loja": "Mercado Livre"
                })
                
                print(f"Encontrado: {nome[:50]}... | Valor: {valor}")
                logging.info(
                    f"[Mercado Livre] Produto encontrado | "
                    f"Nome: {nome} | "
                    f"Preço: {valor}"
                )
                
            except Exception as e:
                erro = f"Erro ao processar item {i}: {str(e)}"
                print(erro)
                logging.error(erro)

    except Exception as e:
        erro = f"Erro na busca '{item_pesquisa}': {str(e)}"
        print(erro)
        logging.error(erro)
        time.sleep(3)

print("\n>>> Pesquisa concluída. Indo para próxima etapa...")


>>> Pesquisando por: Caixa de Som JBL Flip 6...
Capturando nomes e links...
Capturando preços...
Encontrado: Alto-falante portátil Jbl Flip 7 Red... | Valor: R$ 789,30
Encontrado: Caixa De Som Bluetooth 30w Prova D Água Flip 6 Jbl... | Valor: R$ 750,00
Encontrado: Caixa De Som Jbl Flip 6 Bluetooth Potência 30w Pre... | Valor: R$ 928,08

>>> Pesquisando por: Smart TV Samsung 50 polegadas Crystal UHD...
Capturando nomes e links...
Capturando preços...
Encontrado: Smart Tv Samsung Led 50 Lh50befh4ggxzd Led Crystal... | Valor: R$ 2.999,90
Encontrado: Smart Tv U8600f Crystal Uhd 4k 50 2025 Preto Samsu... | Valor: R$ 3.510,99
Encontrado: Smart Tv 50'' Samsung Lh50bedhvggxzd Led Crystal U... | Valor: R$ 3.405,00

>>> Pesquisa concluída. Indo para próxima etapa...


### Web Scraping - Amazon

In [35]:
logging.info(f"Acessando site: {url_amazon}")
navegador.get(url_amazon)

WebDriverWait(navegador, 10).until(
    EC.element_to_be_clickable(
        (By.ID, "twotabsearchtextbox") 
    )
)

for item_pesquisa in PRODUTOS_ELETRONICOS:
    try:
        logging.info(f"Pesquisando produto: {item_pesquisa}")
        print(f"\n>>> Pesquisando por: {item_pesquisa}...")
        
        time.sleep(0.5)
        
        busca = navegador.find_element("xpath", "//*[@id='twotabsearchtextbox']")
        
        busca.clear()
        
        time.sleep(1)
        
        busca.send_keys(item_pesquisa)
        busca.send_keys(Keys.ENTER)

        WebDriverWait(navegador, 10).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "div[data-component-type='s-search-result']")
            )
        )

        time.sleep(1)

        print("Capturando nomes, preços e links...")
        logging.info("Capturando elementos da página (nomes, preços e links)...")
        
        nomes_elementos = navegador.find_elements("css selector", "h2.a-size-base-plus span")[:3]
        precos_elementos = navegador.find_elements("css selector", "span.a-price")[:3]
        links_elementos = navegador.find_elements("css selector", "a.a-link-normal.s-line-clamp-4")[:3]

        for i in range(len(nomes_elementos)):
            try:
                nome = nomes_elementos[i].text
                
                try:
                    inteiro = precos_elementos[i].find_element("css selector", ".a-price-whole").text
                    centavos = precos_elementos[i].find_element("css selector", ".a-price-fraction").text
                    inteiro = inteiro.replace(",", "").replace(".", "")
                    valor = f"R$ {inteiro},{centavos}"
                except:  
                    valor = "Sem preço"

                link = links_elementos[i].get_attribute("href")

                lista_produtos.append({
                    "Pesquisa": item_pesquisa,
                    "Produto": nome,
                    "Preço": valor,
                    "Link": link,
                    "Loja": "Amazon"
                })
                
                print(f"Encontrado: {nome[:50]}... | Valor: {valor}")
                logging.info(
                    f"[Amazon] Produto encontrado | "
                    f"Nome: {nome[:50]}... | "
                    f"Preço: {valor}"
                )
                
            except Exception as e:
                erro = f"Erro ao processar item {i}: {str(e)}"
                print(erro)
                logging.error(erro)

    except Exception as e:
        erro = f"Erro na busca '{item_pesquisa}': {str(e)}"
        print(erro)
        logging.error(erro)
        time.sleep(3)

print("\n>>> Pesquisa concluída. Salvando dados...")
logging.info("Pesquisa concluída. Iniciando processamento e salvamento dos dados.")

if lista_produtos:
    df = pd.DataFrame(lista_produtos)

    print("\n" + "="*60)
    print("PRÉVIA DOS DADOS COLETADOS:")
    print(df.head())

    df.to_csv("resultados.csv", index=False, encoding='utf-8-sig', sep=';')

    print("\n" + "="*60)
    print(f"ARQUIVO 'resultados.csv' GERADO COM SUCESSO!")
    print("="*60)
    logging.info("Arquivo CSV 'resultados.csv' gerado e salvo com sucesso.")
else:
    print("Nenhum produto foi capturado.")
    logging.warning("Nenhum produto foi capturado durante a execução. O CSV não foi gerado.")

time.sleep(1)
logging.info("Encerrando o navegador.")
navegador.quit()


>>> Pesquisando por: Caixa de Som JBL Flip 6...
Capturando nomes, preços e links...
Encontrado: JBL, Caixa de Som, Grip, Bluetooth, Portátil, À Pr... | Valor: R$ 352,30
Encontrado: Electro-Voice EVIVA12P BR Caixa de Som Ativa, 1000... | Valor: Sem preço
Encontrado: Caixa de Som, Edifier R1000T4, 24W RMS... | Valor: Sem preço

>>> Pesquisando por: Smart TV Samsung 50 polegadas Crystal UHD...
Capturando nomes, preços e links...
Encontrado: Samsung Smart TV 50" Crystal UHD 4K U8100F 2025... | Valor: R$ 2195,10
Encontrado: Smart TV 85" Crystal UHD 4K + Smart TV-43" QLED Fu... | Valor: Sem preço
Encontrado: Samsung Combo Smart TV 75" Crystal UHD 4K U8600F +... | Valor: Sem preço

>>> Pesquisa concluída. Salvando dados...

PRÉVIA DOS DADOS COLETADOS:
                                    Pesquisa  \
0                    Caixa de Som JBL Flip 6   
1                    Caixa de Som JBL Flip 6   
2                    Caixa de Som JBL Flip 6   
3  Smart TV Samsung 50 polegadas Crystal UHD   
4  S